In [0]:
# =====================================================
# PARAMÉTRAGE - Widgets pour exécution via Databricks Job
# Permet de réutiliser ce notebook sur différents catalogs
# (dev/staging/prod) sans modifier le code
# =====================================================

dbutils.widgets.text("catalog_name", "banking_lakehouse", "Catalog Unity Catalog")
dbutils.widgets.text("environment", "dev", "Environnement (dev/staging/prod)")

# Récupération des valeurs (soit celles par défaut, soit celles injectées par le Job)
CATALOG = dbutils.widgets.get("catalog_name")
ENVIRONMENT = dbutils.widgets.get("environment")

print(f"✅ Paramètres reçus : catalog={CATALOG}, environment={ENVIRONMENT}")

In [0]:
# =====================================================
# Notebook : 02_ingest_bronze_autoloader
# Objectif : Ingérer les données brutes (clients, transactions)
#            depuis le Volume landing_files vers les tables Bronze
#            en utilisant Auto Loader (Incremental Load)
# Domaine  : Banking Lakehouse
# =====================================================

from pyspark.sql.functions import current_timestamp, input_file_name, col
from pyspark.sql.types import *

# --- Configuration des chemins (Unity Catalog Volumes) ---
SCHEMA_BRONZE = "bronze"

# Chemins sources (landing zone)
PATH_CLIENTS_LANDING = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/landing_files/clients/"
PATH_TRANSACTIONS_LANDING = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/landing_files/transactions/"

# Chemins des checkpoints Auto Loader (mémoire de ce qui a déjà été lu)
PATH_CHECKPOINT_CLIENTS = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/landing_files/_checkpoints/clients/"
PATH_CHECKPOINT_TRANSACTIONS = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/landing_files/_checkpoints/transactions/"

# Chemins des schema location (mémoire du schéma inféré)
PATH_SCHEMA_CLIENTS = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/landing_files/_schemas/clients/"
PATH_SCHEMA_TRANSACTIONS = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/landing_files/_schemas/transactions/"

# Tables Bronze cibles (Delta, gérées par Unity Catalog)
TABLE_CLIENTS_RAW = f"{CATALOG}.{SCHEMA_BRONZE}.clients_raw"
TABLE_TRANSACTIONS_RAW = f"{CATALOG}.{SCHEMA_BRONZE}.transactions_raw"

print("✅ Configuration chargée avec succès")
print(f"Source clients      : {PATH_CLIENTS_LANDING}")
print(f"Source transactions : {PATH_TRANSACTIONS_LANDING}")
print(f"Table cible clients      : {TABLE_CLIENTS_RAW}")
print(f"Table cible transactions : {TABLE_TRANSACTIONS_RAW}")

In [0]:
# =====================================================
# Ingestion Auto Loader - Table Bronze : clients_raw
# Source : Churn_Modelling.csv (dataset réel Kaggle)
# Avec gestion explicite du Schema Evolution (mergeSchema)
# =====================================================

df_clients_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", PATH_SCHEMA_CLIENTS)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(PATH_CLIENTS_LANDING)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

query_clients = (
    df_clients_stream.writeStream
    .format("delta")
    .option("checkpointLocation", PATH_CHECKPOINT_CLIENTS)
    .option("mergeSchema", "true")   # 🆕 Autorise explicitement l'évolution du schéma Delta
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TABLE_CLIENTS_RAW)
)

query_clients.awaitTermination()

print("✅ Ingestion clients_raw terminée avec succès (avec Schema Evolution)")

In [0]:
# =====================================================
# Vérification - Table Bronze : clients_raw
# =====================================================

df_check = spark.table(TABLE_CLIENTS_RAW)

print(f"📊 Nombre de lignes ingérées : {df_check.count()}")
print(f"📋 Schéma de la table :")
df_check.printSchema()

display(df_check.limit(10))

In [0]:
# =====================================================
# Ingestion Auto Loader - Table Bronze : transactions_raw
# Source : creditcard.csv (dataset réel Kaggle - ULB)
# Volume : ~284 807 lignes
# =====================================================

df_transactions_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", PATH_SCHEMA_TRANSACTIONS)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(PATH_TRANSACTIONS_LANDING)
    # --- Ajout des métadonnées techniques (compatible Unity Catalog) ---
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

query_transactions = (
    df_transactions_stream.writeStream
    .format("delta")
    .option("checkpointLocation", PATH_CHECKPOINT_TRANSACTIONS)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TABLE_TRANSACTIONS_RAW)
)

query_transactions.awaitTermination()

print("✅ Ingestion transactions_raw terminée avec succès")

In [0]:
# =====================================================
# Vérification - Table Bronze : transactions_raw
# =====================================================

df_transactions_check = spark.table(TABLE_TRANSACTIONS_RAW)

print(f"📊 Nombre de lignes ingérées : {df_transactions_check.count()}")
print(f"📋 Schéma de la table :")
df_transactions_check.printSchema()

print("\n📈 Répartition des classes (0 = normal, 1 = fraude) :")
df_transactions_check.groupBy("Class").count().orderBy("Class").show()

print("\n🔎 Vérification de la colonne _rescued_data (doit être vide si données propres) :")
df_transactions_check.filter(col("_rescued_data").isNotNull()).count()

In [0]:
# =====================================================
# Vérification finale - Schema Evolution sur clients_raw
# =====================================================

df_final = spark.table(TABLE_CLIENTS_RAW)

print(f"📊 Nombre total de lignes : {df_final.count()}")
print(f"📋 Nouveau schéma (avec Country_Risk_Level) :")
df_final.printSchema()

print("\n🔎 Lignes historiques (doivent avoir Country_Risk_Level = NULL) :")
df_final.filter(col("CustomerId") < 90000000).select(
    "CustomerId", "Surname", "Country_Risk_Level"
).limit(5).show()

print("\n🆕 Nouvelles lignes du batch2 (doivent avoir Country_Risk_Level renseigné) :")
df_final.filter(col("Country_Risk_Level").isNotNull()).select(
    "CustomerId", "Surname", "Country_Risk_Level", "_source_file"
).show()